# Part 3 — Model Tuning: Optuna HPO for LightGBM

Use Optuna to tune LightGBM hyperparameters on a 2M-row subsample (2024 data only) before fitting the winner on the full 2023–2024 training set.

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import config

optuna.logging.set_verbosity(optuna.logging.WARNING)
config.assert_data_exists()

## 1. Load Features

In [2]:
feat_path = config.DATA_PART3_PROCESSED / "flights_2023_2025_features.parquet"
df = pd.read_parquet(feat_path)

TARGET = "ARR_DEL15"
CAT_COLS = ["ORIGIN", "DEST", "OP_CARRIER"]
FEATURES = [c for c in df.columns if c not in [TARGET, "YEAR"]]

# Subsample 2024 for fast HPO
hpo_df = df[df["YEAR"] == 2024].sample(n=2_000_000, random_state=42)
X_hpo = hpo_df[FEATURES].copy()
y_hpo = hpo_df[TARGET]

for col in CAT_COLS:
    if col in X_hpo.columns:
        X_hpo[col] = X_hpo[col].astype("category")

print(f"HPO sample: {X_hpo.shape}")

HPO sample: (2000000, 17)


## 2. Optuna Objective

In [3]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_jobs": -1,
        "random_state": 42,
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X_hpo, y_hpo):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_hpo.iloc[train_idx], y_hpo.iloc[train_idx])
        prob = model.predict_proba(X_hpo.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y_hpo.iloc[val_idx], prob))
    return np.mean(scores)

In [4]:
study = optuna.create_study(direction="maximize", study_name="lgb_part3")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"Best AUC: {study.best_value:.4f}")
print("Best params:", study.best_params)

  0%|          | 0/40 [00:00<?, ?it/s]

Best AUC: 0.7116
Best params: {'n_estimators': 1000, 'learning_rate': 0.03702022200732458, 'num_leaves': 159, 'min_child_samples': 146, 'feature_fraction': 0.658759571848042, 'bagging_fraction': 0.8875352302954408, 'bagging_freq': 1, 'reg_alpha': 9.45737055800694, 'reg_lambda': 2.2451335027383914}


## 3. Train Final Model with Best Params

In [5]:
train_df = df[df["YEAR"] <= 2024]
test_df  = df[df["YEAR"] == 2025]

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET]
X_test  = test_df[FEATURES].copy()
y_test  = test_df[TARGET]

for col in CAT_COLS:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_test[col]  = X_test[col].astype("category")

best_params = {**study.best_params, "objective": "binary", "metric": "auc",
               "verbosity": -1, "n_jobs": -1, "random_state": 42}

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

Training until validation scores don't improve for 50 rounds


[100]	valid_0's auc: 0.656693


[200]	valid_0's auc: 0.658238


[300]	valid_0's auc: 0.65864


[400]	valid_0's auc: 0.659055


...

[700]	valid_0's auc: 0.659865


[800]	valid_0's auc: 0.660028


[900]	valid_0's auc: 0.660138


[1000]	valid_0's auc: 0.660235
Did not meet early stopping. Best iteration is:
[964]	valid_0's auc: 0.660247


,boosting_type,'gbdt'
,num_leaves,159
,max_depth,-1
,learning_rate,0.03702022200732458
,n_estimators,1000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,146


In [6]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

prob = final_model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

print("=== Tuned LightGBM ===")
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, prob):.4f}")
print(f"F1:       {f1_score(y_test, pred):.4f}")
print(classification_report(y_test, pred))

=== Tuned LightGBM ===
Accuracy: 0.7772


ROC-AUC:  0.6602


F1:       0.0977


              precision    recall  f1-score   support

           0       0.78      0.98      0.87   5344846
           1       0.51      0.05      0.10   1534638

    accuracy                           0.78   6879484
   macro avg       0.65      0.52      0.49   6879484
weighted avg       0.72      0.78      0.70   6879484



## 4. Save Tuned Model

In [7]:
final_model.booster_.save_model(str(config.DATA_PART3_PROCESSED / "lgb_tuned.txt"))
print("Saved lgb_tuned.txt")

Saved lgb_tuned.txt
